# Flow Matching Loss（流匹配损失）

**难度：** Easy

**函数签名：** `flow_matching_loss(model_output, x0, x1, t) -> Tensor`

**参数：**
- `model_output` — 预测速度 v_θ(x_t, t) (B, D)
- `x0` — 噪声样本 (B, D)
- `x1` — 数据样本 (B, D)
- `t` — 时间步 (B,)

**返回：** 标量 MSE 损失

**公式：** `target = x1 - x0`（从噪声到数据的直线方向）
`loss = mean(||model_output - target||²)`

**提示：** 插值点 x_t 在外部已计算，t 在此仅为接口完整性

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def flow_matching_loss(model_output, x0, x1, t):
    # ---- 流匹配的核心思想 ----
    # 扩散模型: 沿复杂噪声调度逐步去噪
    # 流匹配: 沿从噪声到数据的直线路径传输
    #
    # 直线路径: x_t = (1-t) · x0 + t · x1, t ∈ [0, 1]
    #   t=0: x_t = x0 (纯噪声)
    #   t=1: x_t = x1 (数据)
    #
    # 直线路径的速度场: dx/dt = x1 - x0 (常数！)
    # 这就是为什么目标速度是 x1 - x0
    #
    # 损失函数: 让模型预测的速度接近真实速度
    # MSE = mean(||v_θ(x_t, t) - (x1 - x0)||²)

    # ---- 计算目标速度 ----
    # 从噪声指向数据的方向，大小为 x1 - x0
    # 注意: 这是常数向量，不依赖于 t！
    target_velocity = x1 - x0

    # ---- MSE 损失 ----
    # diff = 预测速度 - 目标速度
    # diff * diff = 逐元素平方
    # .mean() = 对所有元素取平均
    diff = model_output - target_velocity
    return (diff * diff).mean()

In [ ]:
torch.manual_seed(0)

B, D = 16, 8
x0 = torch.randn(B, D)
x1 = torch.randn(B, D)
t  = torch.rand(B)

# Perfect prediction => loss should be exactly 0
perfect_output = x1 - x0
loss_perfect = flow_matching_loss(perfect_output, x0, x1, t)
print(f"Perfect prediction => loss = {loss_perfect.item():.6f}  (expected 0.0)")

# Random prediction => loss should be positive
random_output = torch.randn(B, D)
loss_random = flow_matching_loss(random_output, x0, x1, t)
print(f"Random prediction  => loss = {loss_random.item():.4f}   (expected > 0)")

# Slightly off prediction => loss between 0 and random
noisy_output = perfect_output + 0.1 * torch.randn(B, D)
loss_noisy = flow_matching_loss(noisy_output, x0, x1, t)
print(f"Noisy prediction   => loss = {loss_noisy.item():.4f}   (expected small but > 0)")

In [ ]:
from torch_judge import check
check("flow_matching")

## 📝 核心思路总结

1. **流匹配 = 直线路径传输**：从噪声到数据走直线，比扩散模型的曲线路径更简单
2. **目标速度 = x1 - x0**：常数向量，不依赖时间步 t
3. **MSE 损失**：预测速度与目标速度的均方误差
4. **t 不参与计算**：损失本身不需要 t，t 仅用于接口一致性